# Проверки витрины за февраль

Целевая витрина:
`sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart`

Проверки:
1. Список колонок
2. Sample за месяц
3. Количество строк и уникальных договоров
4. Количество уникальных ИНН (прямо из витрины или через join)
5. Заполняемость тарифа (с учетом `0` как пустого)
6. Список строк с пустым тарифом

In [ ]:
import sys
from pathlib import Path
from rail_connectors.connection import connect

In [ ]:
# Путь к секретам
ROOT = Path("E:/DTB(dashbord)")
SECRETS_DIR = ROOT / "table_check"
if str(SECRETS_DIR) not in sys.path:
    sys.path.append(str(SECRETS_DIR))

from connection_secrets import LAKE_USER, LAKE_PASSWORD

In [ ]:
def get_impala_connection():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={
            "user_name": LAKE_USER,
            "password": LAKE_PASSWORD,
        },
    )
    imp._init_connection()
    return imp

In [ ]:
imp = get_impala_connection()
print("Impala connection initialized")

In [ ]:
month_start = "2026-02-01"  # YYYY-MM-01
datamart_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart"

## 1) Колонки витрины

In [ ]:
with imp:
    dm0 = imp.fetch(f"select * from {datamart_table} limit 0")

dm_cols = [c.lower() for c in dm0.columns]
print("columns_count =", len(dm_cols))
print(dm0.columns.tolist())

## 2) Sample за февраль

In [ ]:
with imp:
    sample_feb = imp.fetch(f"""
        select *
        from {datamart_table}
        where trx_month = cast('{month_start}' as date)
        limit 100
    """)
sample_feb

## 3) Количество строк и уникальных договоров

In [ ]:
with imp:
    cnt_feb = imp.fetch(f"""
        select
            count(*) as rows_cnt,
            count(distinct agr_id) as uniq_agr_cnt
        from {datamart_table}
        where trx_month = cast('{month_start}' as date)
    """)
cnt_feb

## 4) Уникальные ИНН (автовыбор: из витрины или через join)

Если в витрине есть колонка `inn`, считаем напрямую.
Если нет, считаем через join `datamart.agr_id -> merchants.agr_id`.

In [ ]:
if "inn" in dm_cols:
    with imp:
        inn_cnt = imp.fetch(f"""
            select count(distinct inn) as unique_inn_cnt
            from {datamart_table}
            where trx_month = cast('{month_start}' as date)
        """)
else:
    with imp:
        inn_cnt = imp.fetch(f"""
            select count(distinct m.inn) as unique_inn_cnt
            from {datamart_table} d
            left join sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants m
              on m.agr_id = d.agr_id
            where d.trx_month = cast('{month_start}' as date)
        """)
inn_cnt

## 5) Заполняемость тарифа (0 считаем пустым)

Автовыбор колонки тарифа: ищем в витрине среди популярных имен.
Приоритет: `tariff_plan_id`, `acq_tar_id`, `tariff_id`, `tariff`.

In [ ]:
tariff_candidates = ["tariff_plan_id", "acq_tar_id", "tariff_id", "tariff"]
tariff_col = next((c for c in tariff_candidates if c in dm_cols), None)
print("detected tariff column:", tariff_col)

if tariff_col is None:
    print("Тарифная колонка не найдена в витрине. Проверьте dm0.columns.")
else:
    with imp:
        tariff_fill = imp.fetch(f"""
            select
                count(*) as rows_cnt,
                sum(case
                        when {tariff_col} is null
                          or trim(cast({tariff_col} as string)) in ('', '0', '0.0', '0.00', '0.000', '0.0000')
                        then 1 else 0
                    end) as tariff_empty_cnt,
                sum(case
                        when {tariff_col} is not null
                          and trim(cast({tariff_col} as string)) not in ('', '0', '0.0', '0.00', '0.000', '0.0000')
                        then 1 else 0
                    end) as tariff_filled_cnt,
                round(
                    100.0 * sum(case
                                    when {tariff_col} is not null
                                      and trim(cast({tariff_col} as string)) not in ('', '0', '0.0', '0.00', '0.000', '0.0000')
                                    then 1 else 0
                                 end) / count(*),
                    2
                ) as tariff_fill_rate_pct
            from {datamart_table}
            where trx_month = cast('{month_start}' as date)
        """)
    tariff_fill

## 6) Строки с пустым тарифом (для разбора)

In [ ]:
if tariff_col is None:
    print("Тарифная колонка не определена")
else:
    with imp:
        tariff_null_rows = imp.fetch(f"""
            select *
            from {datamart_table}
            where trx_month = cast('{month_start}' as date)
              and (
                    {tariff_col} is null
                    or trim(cast({tariff_col} as string)) in ('', '0', '0.0', '0.00', '0.000', '0.0000')
                  )
            limit 500
        """)
    tariff_null_rows